In [71]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

examsFolder = "../data/cloudpractitioner"
exam1 = ".backend/data/cloudpractitioner/exam_1.json"


In [72]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader


loader_kwargs = {
    "jq_schema": ".[]", #iterate over question objects
    "text_content": False
}


loader = DirectoryLoader(
    path="../data/cloudpractitioner/", 
    glob="**/*.json", #ensures json
    loader_cls=JSONLoader,
    loader_kwargs=loader_kwargs
)

docs = loader.load()

print(f"Loaded {len(docs)} questions from the folder.")

Loaded 991 questions from the folder.


In [76]:
import json

sanitized_docs = []

for doc in docs:
    # 1. Parse the string inside the 'text' field back into a dictionary
    raw_data = json.loads(doc.page_content)
    
    # 2. Map the actual values to the metadata keys
    doc.metadata = {
        "choices": raw_data.get("choices"),
        "category": raw_data.get("category"),
        "answer": raw_data.get("answer"),
        "difficulty": raw_data.get("difficulty"),
    }
    
    # 3. Use the actual question text as the page_content for the vector search
    doc.page_content = raw_data.get("question")
    
    sanitized_docs.append(doc)


In [78]:
#text split
print(docs[0].page_content)
print(docs[0].metadata)
print("variable docs length  " + str(len(docs))) 

What time-savings advantage is offered with the use of Amazon Rekognition?
{'choices': ['A. Amazon Rekognition provides automatic watermarking of images.', 'B. Amazon Rekognition provides automatic detection of objects appearing in pictures.', 'C. Amazon Rekognition provides the ability to resize millions of images automatically.', 'D. Amazon Rekognition uses Amazon Mechanical Turk to allow humans to bid on object detection jobs.'], 'category': 'Machine Learning', 'answer': 'B', 'difficulty': 2}
variable docs length  991


### Embedding

In [79]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [80]:
#embedding test
vector_1 = embeddings.embed_query(docs[0].page_content)

print(f"Generated vector of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vector of length 1536

[0.0374404639005661, 0.0011903299018740654, 0.05431877076625824, 0.004493457730859518, 0.006717613898217678, 0.017432088032364845, -0.01557812187820673, 0.04613243043422699, -0.010257001966238022, 0.01922585815191269]


In [81]:
from langchain_community.vectorstores.upstash import UpstashVectorStore
import os

os.environ["UPSTASH_VECTOR_REST_URL"] = "https://loving-kingfish-56853-us1-vector.upstash.io"
os.environ["UPSTASH_VECTOR_REST_TOKEN"] = ""

store = UpstashVectorStore(
    embedding=embeddings
)

### Adding documents Here

In [82]:
#ids = store.add_documents(documents=sanitized_docs)

## Similarity search

In [88]:
#find similar questions 
results = store.similarity_search(
    "AWS allows users to manage their resources using a web based user interface."
)

print(len(results))
for result in results:
    print(result)

4
page_content='AWS allows users to manage their resources using a web based user interface. What is the name of this interface?' metadata={'choices': ['A. AWS CLI.', 'B. AWS API.', 'C. AWS SDK.', 'D. AWS Management Console.'], 'category': 'Cloud Concepts', 'answer': 'D', 'difficulty': 2}
page_content='What is the AWS tool that enables you to use scripts to manage all AWS services and resources?' metadata={'choices': ['A. AWS Console.', 'B. AWS Service Catalog.', 'C. AWS OpsWorks.', 'D. AWS CLI.'], 'category': 'Cloud Concepts', 'answer': 'D', 'difficulty': 2}
page_content='AWS CloudFormation is designed to help the user:' metadata={'choices': ['A. model and provision resources.', 'B. update application code.', 'C. set up data lakes.', 'D. create reports for billing.'], 'category': 'Management & Governance', 'answer': 'A', 'difficulty': 2}
page_content='Which of the following services allows customers to manage their agreements with AWS?' metadata={'choices': ['A. AWS Artifact.', 'B. AW